# Train our own Acehnese → Indonesian model (free Colab)
**Before you run:** Runtime → Change runtime type → **T4 GPU**. Free, no API key. First run downloads the NLLB model (~2.5 GB) from Hugging Face.

What this does: fine-tunes Meta's NLLB-200 (600M, supports Acehnese) on our pairs, keeps the best epoch on the NusaX *validation* set, then scores base vs fine-tuned **once** on the NusaX *test* set.
Training takes roughly 10–30 minutes on a T4 with ~500 pairs.

In [ ]:
!git clone -b claude/dataset-product-build-108xu8 https://github.com/Mutaib-yye/aceh_code
%cd aceh_code
!pip install -q -r requirements.txt transformers sentencepiece torch

In [ ]:
!python -m acehid fetch-nusax      # evaluation CSVs (CC-BY-SA)
!nvidia-smi | head -5

In [ ]:
# Optional: add reviewed hikayat pairs (upload data/final/train.json made by `python -m acehid export` first)
EXTRA = []   # e.g. ["data/final/train.json"]
!python -m acehid train --epochs 5 --extra {' '.join(EXTRA)}

In [ ]:
import json; r = json.load(open('models/ace-id-nllb/training_log.json'))
print('train pairs:', r['train_pairs'], '| best epoch:', r.get('best_epoch'))
print('TEST base      :', r['test_base'])
print('TEST fine-tuned:', r['test_finetuned'])

In [ ]:
# try it
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained('models/ace-id-nllb', src_lang='ace_Latn'); m = AutoModelForSeq2SeqLM.from_pretrained('models/ace-id-nllb').to('cuda')
x = tok(['Tiep uroe tuhan jijak u pasi jimeuen ngon anoe'], return_tensors='pt').to('cuda')
print(tok.batch_decode(m.generate(**x, forced_bos_token_id=tok.convert_tokens_to_ids('ind_Latn'), num_beams=4, max_new_tokens=64), skip_special_tokens=True))

In [ ]:
# download the trained model (keep it: it is YOUR model)
!zip -qr ace-id-nllb.zip models/ace-id-nllb
from google.colab import files; files.download('ace-id-nllb.zip')